# Ask 1 — The Model Has Never Read Your Stuff

Three ways to ask the same question: from memory (a guess), with the whole
pile pasted in (cost and dilution), with hand-picked context (the answer).
The hand-picking is what lessons 2–4 automate.

**No class API key?** The arithmetic runs offline; the live asks have
precomputed outputs.

In [ ]:
# The pile: documents from the (fictional) Jefferson High School.
# Real enough to search, small enough to read whole.
PILE = {
 "handbook_academics": """S4.1 Grading scale. A 90-100, B 80-89, C 70-79, D 60-69.
Semester grades weight exams at 30 percent.
S4.2 Exam Retake Policy. This policy applies to final exams only. Students
receive one retake per semester, requested within ten school days. The
higher score stands.
S4.3 Grade appeals. Appeals go to the department head in writing within
fifteen school days of the posted grade.
S4.5 Late work. Assignments lose 10 percent per school day late, to a
maximum of 50 percent. Teachers may grant extensions for documented
emergencies.""",
 "handbook_schedule": """S2.0 Bell schedule. Regular days run eight periods,
8:15 AM to 3:20 PM.
S2.1 Wednesday schedule. Dismissal at 1:30 PM every Wednesday for staff
development.
S2.4 Late arrival. Students arriving after 8:30 AM sign in at the main
office with a note.""",
 "handbook_trips": """S5.1 Field trips require a signed permission form
submitted five school days in advance.
S5.2 Trip costs above 20 dollars qualify for the student activity fund.
S5.4 Chaperones must be approved district volunteers.""",
 "handbook_athletics": """S6.2 Eligibility. Athletes must hold a C average
during their season. Freshmen may try out for varsity teams.
S6.3 Petitions. A varsity roster spot for a freshman requires a coach's
petition to the athletic director.""",
 "robotics_minutes": """Robotics club meets Tuesdays in room 214. Regional
trip is April 18; bring your signed permission form by April 10. Dues are
15 dollars for the year.""",
 "clubs_list": """Active clubs: robotics (Tuesdays), debate (Thursdays),
art collective (Fridays), chess (lunch, library). Sign-up forms at the
student office.""",
 "bus_routes": """Routes 12 and 15 serve the north side. Final pickup at
4:45 PM outside door C. Activity buses run Tuesday and Thursday only.""",
 "cafeteria": """Lunch periods run 11:10, 11:55, and 12:40. Breakfast is
served from 7:40 AM. Menus post monthly on the food services page.""",
}
print(f"{len(PILE)} documents, {sum(len(t) for t in PILE.values())} characters total")

In [ ]:
%pip install -q anthropic

In [ ]:
import os, getpass
# Ask your teacher for the class API key. getpass keeps it out of the file.
try:
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Class API key: ")
    HAVE_KEY = len(os.environ["ANTHROPIC_API_KEY"]) > 10
except Exception:
    HAVE_KEY = False
print("Key loaded." if HAVE_KEY else "No key - precomputed outputs shown below each live cell.")

In [ ]:
MODEL = "claude-opus-5"

def llm(prompt, max_tokens=800):
    import anthropic
    client = anthropic.Anthropic()
    return client.messages.create(model=MODEL, max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}]).content[-1].text

## Strategy 1 — memory only

Nothing from the pile. The model answers from what *other* schools'
handbooks tend to say — fluent, specific, and unanchored.

In [ ]:
QUESTION = "Can I retake a final at Jefferson High, and what are the rules?"

if HAVE_KEY:
    print(llm(QUESTION))
else:
    print("""Precomputed (no documents provided):
'Most high schools allow final exam retakes if you scored below 70%,
typically within two weeks of the exam...'

Jefferson's actual rule (S4.2): below NO threshold, ONE per semester,
TEN school days. Every specific in the guess is wrong for THIS school -
and nothing about the answer's tone warns you.""")

## Strategy 2 — paste everything

It works on a pile this small. The arithmetic below is why it stops
working: context is paid working memory, and relevance dilutes.

In [ ]:
pile_text = "\n\n".join(f"== {name} ==\n{text}" for name, text in PILE.items())
pile_chars = len(pile_text)
relevant_chars = len(PILE["handbook_academics"].split("S4.3")[0].split("S4.2")[1])

print(f"characters sent:      {pile_chars:6d}")
print(f"characters relevant:  {relevant_chars:6d}  ({100*relevant_chars/pile_chars:.1f}%)")
print(f"cost multiplier vs sending only the relevant part: {pile_chars/relevant_chars:.0f}x")
print()
print("Now scale the pile: a real 200-page handbook is ~400,000 characters -")
print("most models' working memory fills, you pay for every character, and")
print("the one relevant paragraph competes with 199 pages of noise.")

if HAVE_KEY:
    print(llm(f"Answer from these documents only:\n{pile_text}\n\nQ: {QUESTION}"))

## Strategy 3 — retrieve first (by hand, today)

In [ ]:
handpicked = PILE["handbook_academics"].split("S4.3")[0]   # S4.1 + S4.2
prompt = f"""Answer using ONLY the sections below. Cite the section.

{handpicked}

Q: {QUESTION}"""
print(f"characters sent: {len(prompt)} ({100*len(prompt)/pile_chars:.0f}% of the pile)\n")
if HAVE_KEY:
    print(llm(prompt))
else:
    print("""Precomputed:
'Yes - finals only, one retake per semester, requested within ten school
days; the higher score stands [S4.2].'

Cheap, focused, checkable. Finding those sections AUTOMATICALLY - for any
question - is the machine lessons 2-4 build.""")

## Try it

1. Ask strategy 1 about YOUR school. Which specifics did it invent?
2. Compute the cost multiplier for a pile ten times this size.
3. **Build turn-in:** your chosen pile's inventory and your three
   questions, each with a guess at which document holds the answer.